# 00 — Environment and status
Checks the runtime (LOCAL / KAGGLE / CAMBER), paths, optional dependencies,
Camber CLI/auth and shows what state files prove so far.
Nothing here downloads or changes data.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
import importlib
mods = {"yaml": "core", "pandas": "core", "PIL": "core", "imagehash": "stage", "cv2": "stage",
        "fiftyone": "stage", "gdown": "stage", "huggingface_hub": "stage", "ultralytics": "train", "torch": "train"}
rows = []
for m, extra in mods.items():
    try:
        mod = importlib.import_module(m); rows.append((m, extra, getattr(mod, "__version__", "ok")))
    except Exception as e:
        rows.append((m, extra, f"MISSING ({type(e).__name__})"))
display(pd.DataFrame(rows, columns=["module", "extra", "version"]))

In [ ]:
from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
cfg = load_config("yolo11n_ablation")
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
print("camber binary:", cam.binary)
print("camber usable:", cam.available(), "(key from env/Kaggle secret or `camber login`)")
print("config hash:", cfg.config_hash()[:12])

In [ ]:
from yolov11sdi.reporting import status
cfg = load_config("yolo11n_ablation")
display(status.stage_table(cfg, paths))
ov = status.overview(cfg, paths)
display(pd.Series({k: v for k, v in ov.items() if k != "disk"}, dtype=object).to_frame("value"))
d = ov["disk"]; print(f"disk free {d['free_gb']:.1f} / {d['total_gb']:.1f} GB at {d['path']}")